In [12]:
# --- Imports and global setup ---
import cv2
import numpy as np
import threading
import queue
import time
import sys
from IPython.display import clear_output
from deepface import DeepFace

# --- Globals ---
MODEL_NAME = "Facenet"  # or "VGG-Face", "ArcFace"
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
frame_q = queue.Queue(maxsize=2)
result_q = queue.Queue(maxsize=2)

cv2.destroyAllWindows()
print("✅ Environment ready")


✅ Environment ready


In [5]:

def build_embeddings(db_path="database/", model_name="Facenet"):
    """
    Builds face embeddings for all valid images inside the database directory.
    Skips hidden files (like macOS ._ files) and unreadable images automatically.
    """
    print(f"🔍 Building embeddings from '{db_path}'...")
    embeddings = []
    valid_exts = (".jpg", ".jpeg", ".png", ".bmp")

    # Walk through the database directory
    for root, dirs, files in os.walk(db_path):
        for file in files:
            # Skip hidden/system files
            if file.startswith("._") or file.startswith(".") or not file.lower().endswith(valid_exts):
                continue

            path = os.path.join(root, file)
            name = os.path.splitext(os.path.basename(file))[0]

            try:
                reps = DeepFace.represent(img_path=path, model_name=model_name, enforce_detection=False)
                for rep in reps:
                    embeddings.append({
                        "name": name,
                        "embedding": rep["embedding"]
                    })
                print(f"✅ Added: {name}")
            except Exception as e:
                print(f"⚠️ Skipping {file}: {e}")

    if not embeddings:
        print("⚠️ No valid embeddings found. Add clear face images to the database folder.")
    else:
        print(f"✅ Loaded {len(embeddings)} valid embeddings.")
    return embeddings

In [7]:
def find_best_match(query_emb, embeddings, threshold=THRESHOLD):
    """
    Compare query_emb to precomputed embeddings list, return (name, distance) or (None, inf).
    """
    if query_emb is None or not embeddings:
        return None, float("inf")
    dists = []
    for entry in embeddings:
        db_emb = entry["embedding"]
        dist = np.linalg.norm(query_emb - db_emb)
        dists.append((entry["name"], dist))
    dists.sort(key=lambda x: x[1])
    name, dist = dists[0]
    if dist <= threshold:
        return name, dist
    return None, dists[0][1]

NameError: name 'THRESHOLD' is not defined

In [8]:
# --- Face recognition worker thread ---
def processing_worker(model_name, embeddings):
    print("🧠 Worker started with model:", model_name)
    while True:
        item = frame_q.get()
        if item is None:  # stop signal
            break

        frame_id, frame = item
        results = []

        try:
            detections = DeepFace.represent(frame, model_name=model_name, enforce_detection=False)
            for det in detections:
                bbox = det.get("facial_area", None)
                embedding = det["embedding"]

                # simple distance-based match
                best_match, best_dist = None, 1e9
                for e in embeddings:
                    dist = np.linalg.norm(np.array(e["embedding"]) - np.array(embedding))
                    if dist < best_dist:
                        best_match, best_dist = e["name"], dist

                results.append({
                    "bbox": list(bbox.values()) if bbox else [0, 0, 0, 0],
                    "name": best_match if best_dist < 0.9 else None,
                    "distance": best_dist
                })
        except Exception as e:
            print("Worker error:", e)

        result_q.put((frame_id, results))
    print("🧵 Worker stopped")


In [ ]:
# Ensure queues exist
if 'frame_q' not in globals(): frame_q = queue.Queue(maxsize=2)
if 'result_q' not in globals(): result_q = queue.Queue(maxsize=2)

FRAME_WIDTH, FRAME_HEIGHT = 640, 480
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

if not cap.isOpened():
    print("❌ Cannot open webcam")
    cap.release()
    cv2.destroyAllWindows()
else:
    print("🎥 Camera started. Click the preview window and press 'q' to quit.")

    last_time = time.time()
    frame_id = 0
    fps_smooth = 0.0
    key = -1  # initialize properly

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("⚠️ Frame grab failed")
                break

            frame_id += 1

            if not frame_q.full():
                frame_q.put_nowait((frame_id, frame.copy()))

            # get latest results
            latest = None
            while not result_q.empty():
                latest = result_q.get_nowait()
            results = latest[1] if latest else []

            # draw results
            for res in results:
                bbox = res.get("bbox")
                if not bbox or any(v is None for v in bbox):
                    continue
                try:
                    x1, y1, x2, y2 = map(int, bbox)
                except Exception:
                    continue

                name = res.get("name") or "Unknown"
                dist = res.get("distance", 0.0)
                color = (0, 255, 0) if res.get("name") else (0, 0, 255)
                label = f"{name} {dist:.2f}" if res.get("name") else "Unknown"

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(10, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # fps display
            now = time.time()
            fps = 1.0 / (now - last_time) if now != last_time else 0
            last_time = now
            fps_smooth = 0.9 * fps_smooth + 0.1 * fps if fps_smooth else fps
            cv2.putText(frame, f"FPS {fps_smooth:.1f}", (10, 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            cv2.imshow("Face Recognition (press q to quit)", frame)

            # WaitKey bug fix for Jupyter
            key = cv2.waitKeyEx(1)
            if key != -1:
                if key & 0xFF == ord('q'):
                    print("🛑 Quitting...")
                    break

    except KeyboardInterrupt:
        print("🛑 Interrupted manually")

    finally:
        cap.release()
        frame_q.put(None)
        cv2.destroyAllWindows()
        clear_output(wait=True)
        print("✅ Clean exit")


🎥 Camera started. Click the preview window and press 'q' to quit.
🛑 Quitting...


KeyboardInterrupt: 

: 